# Supply Chain Synthetic Dataset Generator

By: Edrian Abagat and Isaiah Mariano

This notebook generates synthetic datasets that mimic the **Kaggle Supply Chain Logistics Problem** dataset structure:

| Dataset | Description |
|---|---|
| `OrderList` | Individual orders with routing, service levels, quantities, and weights |
| `FreightRates` | Carrier×route×weight-bracket rate table |
| `WhCosts` | Warehouse (plant) cost per unit |
| `WhCapacities` | Warehouse (plant) daily capacity |
| `ProductsPerPlant` | Product–plant assignment |
| `VmiCustomers` | VMI customer–plant relationships |
| `PlantPorts` | Plant–port connectivity |

All seven DataFrames are stored in a `tables` dict and are ready for EDA.


## 1. Imports

In [ ]:
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Display options
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("Imports OK")

Imports OK


## 2. Configuration

Edit `GeneratorConfig` to control scale, time range, VMI settings, and **all statistical distributions**.

### Distribution parameters guide

| Parameter | What it controls |
|---|---|
| `order_qty_dist` | `"uniform"` → U(min, max); `"lognormal"` → right-skewed; `"zipf"` → heavy-tailed |
| `weight_per_unit_dist` | How unit weight is drawn: `"uniform"` or `"gamma"` |
| `service_level_probs` | Fraction of orders with each service level (DTP / DTD / CRF) |
| `carrier_mode_probs` | Probability a carrier operates in AIR / GROUND / SEA |
| `wh_cost_dist` | Warehouse cost/unit distribution: `"uniform"` or `"lognormal"` |
| `weight_brackets_kg` | Upper bounds (kg) for freight-rate weight tiers |
| `tpt_probs` | Probability mass function over TPT (0..4 days) |
| `ship_ahead_probs` | PMF over ship-ahead day counts (0..6) |
| `ship_late_probs` | PMF over ship-late day counts (0..6) |
| `carrier_multimodal_prob` | Probability a carrier has more than one transport mode |
| `ports_per_carrier_range` | Min/max origin ports covered by a carrier |


In [ ]:
OutputFormat = Literal["csv", "parquet"]


@dataclass
class GeneratorConfig:
    """Full configuration for the supply chain synthetic data generator."""

    # ── Scale ─────────────────────────────────────────────────────────────────
    n_orders: int = 10_000
    n_customers: int = 500
    n_products: int = 200
    n_plants: int = 20
    n_ports: int = 8
    n_carriers: int = 12

    # ── Time range ────────────────────────────────────────────────────────────
    start_date: str = "2026-01-01"
    end_date: str = "2026-01-02"

    # ── VMI settings ──────────────────────────────────────────────────────────
    # VMI plant IDs must be valid PLANT_XXX labels within n_plants.
    # Pass None to auto-select ~20 % of plants as VMI.
    vmi_plant_ids: Optional[Tuple[str, ...]] = None
    vmi_customer_ratio: float = 0.15  # fraction of customers that are VMI

    # ── Product–plant coverage ────────────────────────────────────────────────
    min_products_per_plant: float = 0.30  # each plant carries ≥30 % of catalogue
    max_products_per_plant: float = 0.70  # each plant carries ≤70 % of catalogue

    # ── Capacity buffer ───────────────────────────────────────────────────────
    # Plant daily capacity = peak assigned quantity × (1 + buffer)
    capacity_buffer: float = 0.20

    # ── Order quantity distribution ───────────────────────────────────────────
    # "uniform" | "lognormal" | "zipf"
    order_qty_dist: str = "uniform"
    min_order_qty: int = 5
    max_order_qty: int = 50
    # lognormal parameters (used when order_qty_dist="lognormal")
    order_qty_lognormal_mean: float = 3.5   # log-scale mean (exp ≈ 33)
    order_qty_lognormal_sigma: float = 0.8
    # zipf parameter (used when order_qty_dist="zipf")
    order_qty_zipf_a: float = 1.5

    # ── Weight per unit ───────────────────────────────────────────────────────
    # "uniform" | "gamma"
    weight_per_unit_dist: str = "uniform"
    weight_per_unit_min: float = 0.1   # kg / unit (uniform)
    weight_per_unit_max: float = 5.0   # kg / unit (uniform)
    weight_per_unit_gamma_shape: float = 2.0  # gamma shape (k)
    weight_per_unit_gamma_scale: float = 1.5  # gamma scale (θ) → mean = k×θ = 3 kg

    # ── Service level probabilities ───────────────────────────────────────────
    # Must sum to 1.0; order: [DTP, DTD, CRF]
    service_level_probs: Tuple[float, float, float] = (0.67, 0.23, 0.10)

    # ── TPT (transport days in system) distribution ───────────────────────────
    # PMF over [0, 1, 2, 3, 4]; must sum to 1.0
    tpt_probs: Tuple[float, ...] = (0.05, 0.23, 0.70, 0.01, 0.01)

    # ── Ship-ahead / ship-late day counts ─────────────────────────────────────
    # PMF over [0, 1, 2, 3, 4, 5, 6]; must sum to 1.0
    ship_ahead_probs: Tuple[float, ...] = (0.25, 0.05, 0.05, 0.40, 0.10, 0.10, 0.05)
    ship_late_probs: Tuple[float, ...] = (0.95, 0.01, 0.01, 0.01, 0.01, 0.005, 0.005)

    # ── Carrier mode assignment ────────────────────────────────────────────────
    # Probability carrier is assigned AIR / GROUND / SEA (before multi-modal).
    carrier_mode_probs: Tuple[float, float, float] = (0.50, 0.30, 0.20)
    # Probability a carrier operates multiple modes
    carrier_multimodal_prob: float = 0.25

    # ── Carrier port coverage ─────────────────────────────────────────────────
    ports_per_carrier_range: Tuple[int, int] = (1, 4)  # how many origin ports

    # ── Freight rate parameters ────────────────────────────────────────────────
    # Weight bracket upper bounds (kg). Final open bracket is added automatically.
    weight_brackets_kg: Tuple[float, ...] = (5, 10, 25, 50, 100, 300, 500, 1000, 5000)
    # Rate (cost/kg) range per mode
    air_rate_range: Tuple[float, float] = (0.50, 5.00)   # $/kg for air
    ground_rate_range: Tuple[float, float] = (0.05, 0.80)  # $/kg for ground
    sea_rate_range: Tuple[float, float] = (0.02, 0.30)   # $/kg for sea
    # Minimum cost (flat fee) range per mode
    air_min_cost_range: Tuple[float, float] = (30.0, 120.0)
    ground_min_cost_range: Tuple[float, float] = (5.0, 30.0)
    sea_min_cost_range: Tuple[float, float] = (50.0, 500.0)
    # Transit days per mode (mean, std)
    air_tpt_days: Tuple[float, float] = (2.0, 1.0)     # mean, std
    ground_tpt_days: Tuple[float, float] = (3.0, 1.0)
    sea_tpt_days: Tuple[float, float] = (20.0, 5.0)

    # ── Warehouse cost distribution ────────────────────────────────────────────
    # "uniform" | "lognormal"
    wh_cost_dist: str = "lognormal"
    wh_cost_min: float = 0.30   # $/unit  (uniform)
    wh_cost_max: float = 2.50   # $/unit  (uniform)
    wh_cost_lognormal_mean: float = 0.0   # log-scale mean → exp(0)≈$1/unit
    wh_cost_lognormal_sigma: float = 0.6

    # ── Warehouse capacity distribution ───────────────────────────────────────
    # Daily capacity is derived from order load + buffer, but has a floor.
    wh_capacity_floor: int = 5   # minimum daily capacity (units)

    # ── Output ────────────────────────────────────────────────────────────────
    output_dir: str = "supply_chain_output"
    output_format: OutputFormat = "csv"
    seed: int = 42

    # ── Internal ID prefixes ──────────────────────────────────────────────────
    plant_prefix: str = "PLANT"
    port_prefix: str = "PORT"
    carrier_prefix: str = "CARRIER"
    customer_prefix: str = "CUST"
    product_id_start: int = 1_600_000  # product IDs are 7-digit integers

    def __post_init__(self):
        """Validation runs automatically after __init__."""
        self._validate()

    def _validate(self):
        """Raise ValueError for any invalid configuration."""
        errors = []

        # Scale
        for attr in ("n_orders", "n_customers", "n_products", "n_plants", "n_ports", "n_carriers"):
            v = getattr(self, attr)
            if not isinstance(v, int) or v < 1:
                errors.append(f"{attr} must be a positive integer, got {v!r}")

        # Date range
        try:
            s = pd.to_datetime(self.start_date)
            e = pd.to_datetime(self.end_date)
            if s >= e:
                errors.append("start_date must be before end_date")
        except Exception:
            errors.append("start_date / end_date must be parseable date strings (YYYY-MM-DD)")

        # VMI ratio
        if not (0.0 <= self.vmi_customer_ratio <= 1.0):
            errors.append(f"vmi_customer_ratio must be in [0, 1], got {self.vmi_customer_ratio}")

        # Product–plant coverage
        if not (0.0 < self.min_products_per_plant <= self.max_products_per_plant <= 1.0):
            errors.append(
                "Need 0 < min_products_per_plant ≤ max_products_per_plant ≤ 1, "
                f"got [{self.min_products_per_plant}, {self.max_products_per_plant}]"
            )

        # Order qty
        if self.order_qty_dist not in ("uniform", "lognormal", "zipf"):
            errors.append(f"order_qty_dist must be 'uniform', 'lognormal', or 'zipf'")
        if self.min_order_qty < 1 or self.min_order_qty > self.max_order_qty:
            errors.append(f"Need 1 ≤ min_order_qty ≤ max_order_qty")

        # Weight dist
        if self.weight_per_unit_dist not in ("uniform", "gamma"):
            errors.append(f"weight_per_unit_dist must be 'uniform' or 'gamma'")

        # Probability vectors
        def _check_pmf(probs, name, length):
            if len(probs) != length:
                errors.append(f"{name} must have {length} elements, got {len(probs)}")
                return
            if abs(sum(probs) - 1.0) > 1e-6:
                errors.append(f"{name} must sum to 1.0 (got {sum(probs):.6f})")
            if any(p < 0 for p in probs):
                errors.append(f"{name} must have non-negative entries")

        _check_pmf(self.service_level_probs, "service_level_probs", 3)
        _check_pmf(self.tpt_probs, "tpt_probs", 5)
        _check_pmf(self.ship_ahead_probs, "ship_ahead_probs", 7)
        _check_pmf(self.ship_late_probs, "ship_late_probs", 7)
        _check_pmf(self.carrier_mode_probs, "carrier_mode_probs", 3)

        # Carrier
        if not (0.0 <= self.carrier_multimodal_prob <= 1.0):
            errors.append("carrier_multimodal_prob must be in [0, 1]")
        mn, mx = self.ports_per_carrier_range
        if mn < 1 or mn > mx:
            errors.append("ports_per_carrier_range: need 1 ≤ min ≤ max")
        if mx > self.n_ports:
            errors.append(
                f"ports_per_carrier_range max ({mx}) cannot exceed n_ports ({self.n_ports})"
            )

        # VMI plant IDs
        if self.vmi_plant_ids is not None:
            max_idx = self.n_plants
            for pid in self.vmi_plant_ids:
                try:
                    idx = int(pid.split("_")[-1])
                    if idx < 1 or idx > max_idx:
                        errors.append(f"vmi_plant_id {pid!r} index out of range [1, {max_idx}]")
                except (ValueError, IndexError):
                    errors.append(f"vmi_plant_id {pid!r} does not match PLANT_NNN format")

        # Warehouse cost
        if self.wh_cost_dist not in ("uniform", "lognormal"):
            errors.append("wh_cost_dist must be 'uniform' or 'lognormal'")

        # Weight brackets must be strictly increasing
        wb = list(self.weight_brackets_kg)
        if wb != sorted(wb) or len(wb) != len(set(wb)):
            errors.append("weight_brackets_kg must be strictly increasing")
        if any(w <= 0 for w in wb):
            errors.append("weight_brackets_kg values must be positive")

        if errors:
            raise ValueError("GeneratorConfig validation failed:\n  " + "\n  ".join(errors))


# ── Sanity-check the default config ──────────────────────────────────────────
try:
    _cfg_test = GeneratorConfig()
    print("✅  Default GeneratorConfig is valid.")
except ValueError as e:
    print("❌  Default config invalid:", e)

✅  Default GeneratorConfig is valid.


## 3. Generator class

In [ ]:
class SupplyChainDataGenerator:
    """Generates all seven supply-chain tables that mirror the Kaggle dataset."""

    _MODES = ["AIR", "GROUND", "SEA"]
    _SERVICE_LEVELS = ["DTP", "DTD", "CRF"]

    # ── Construction ─────────────────────────────────────────────────────────

    def __init__(self, config: GeneratorConfig):
        self.config = config
        self.rng = np.random.default_rng(config.seed)
        self._build_ids()
        self._resolve_vmi_plants()

    def _build_ids(self):
        cfg = self.config
        self.plant_ids = np.array(
            [f"{cfg.plant_prefix}_{i:02d}" for i in range(1, cfg.n_plants + 1)]
        )
        self.port_ids = np.array(
            [f"{cfg.port_prefix}{i:02d}" for i in range(1, cfg.n_ports + 1)]
        )
        self.carrier_ids = np.array(
            [f"{cfg.carrier_prefix}_{i:02d}" for i in range(1, cfg.n_carriers + 1)]
        )
        self.customer_ids = np.array(
            [f"{cfg.customer_prefix}_{i:03d}" for i in range(1, cfg.n_customers + 1)]
        )
        self.product_ids = np.arange(
            cfg.product_id_start, cfg.product_id_start + cfg.n_products
        )

    def _resolve_vmi_plants(self):
        """Set self.vmi_plant_ids; auto-select ~20 % if not configured."""
        if self.config.vmi_plant_ids is not None:
            self.vmi_plant_ids = np.array(self.config.vmi_plant_ids)
        else:
            n_vmi = max(1, int(self.config.n_plants * 0.20))
            self.vmi_plant_ids = self.rng.choice(self.plant_ids, size=n_vmi, replace=False)

    # ── Helper: sample order quantities ──────────────────────────────────────

    def _sample_quantities(self, n: int) -> np.ndarray:
        cfg = self.config
        if cfg.order_qty_dist == "uniform":
            return self.rng.integers(cfg.min_order_qty, cfg.max_order_qty + 1, size=n)
        elif cfg.order_qty_dist == "lognormal":
            raw = self.rng.lognormal(cfg.order_qty_lognormal_mean, cfg.order_qty_lognormal_sigma, size=n)
            return np.clip(np.round(raw).astype(int), cfg.min_order_qty, cfg.max_order_qty)
        else:  # zipf
            raw = self.rng.zipf(cfg.order_qty_zipf_a, size=n)
            return np.clip(raw, cfg.min_order_qty, cfg.max_order_qty)

    # ── Helper: sample unit weights ───────────────────────────────────────────

    def _sample_unit_weights(self, n: int) -> np.ndarray:
        cfg = self.config
        if cfg.weight_per_unit_dist == "uniform":
            return self.rng.uniform(cfg.weight_per_unit_min, cfg.weight_per_unit_max, size=n)
        else:  # gamma
            return self.rng.gamma(cfg.weight_per_unit_gamma_shape,
                                  cfg.weight_per_unit_gamma_scale, size=n)

    # ── 1. ProductsPerPlant ──────────────────────────────────────────────────

    def generate_products_per_plant(self, vmi_plant_ids: np.ndarray) -> pd.DataFrame:
        """Assign a random product subset to each plant (ProductsPerPlant sheet)."""
        cfg = self.config
        n_prods = len(self.product_ids)
        low = max(1, int(n_prods * cfg.min_products_per_plant))
        high = max(low + 1, int(n_prods * cfg.max_products_per_plant) + 1)

        rows = []
        for plant_id in self.plant_ids:
            size = int(self.rng.integers(low, high))
            chosen = self.rng.choice(self.product_ids, size=size, replace=False)
            for prod in chosen:
                rows.append({"Plant Code": plant_id, "Product ID": prod})

        df = pd.DataFrame(rows).drop_duplicates()

        # Every product covered by ≥1 plant
        covered = set(df["Product ID"])
        for prod in self.product_ids:
            if prod not in covered:
                plant = self.rng.choice(self.plant_ids)
                df = pd.concat([df, pd.DataFrame([{"Plant Code": plant, "Product ID": prod}])],
                               ignore_index=True)

        # Every product covered by ≥1 VMI plant
        prod_to_plants = df.groupby("Product ID")["Plant Code"].apply(set).to_dict()
        vmi_set = set(vmi_plant_ids)
        for prod in self.product_ids:
            if not (prod_to_plants.get(prod, set()) & vmi_set):
                vmi_plant = self.rng.choice(vmi_plant_ids)
                df = pd.concat([df, pd.DataFrame([{"Plant Code": vmi_plant, "Product ID": prod}])],
                               ignore_index=True)

        return df.drop_duplicates().reset_index(drop=True)

    # ── 2. VmiCustomers ──────────────────────────────────────────────────────

    def generate_vmi_customers(self, vmi_plant_ids: np.ndarray) -> pd.DataFrame:
        """Assign a random subset of customers to VMI plants (VmiCustomers sheet)."""
        cfg = self.config
        n_vmi_cust = max(1, int(cfg.n_customers * cfg.vmi_customer_ratio))
        vmi_customers = self.rng.choice(self.customer_ids, size=n_vmi_cust, replace=False)

        rows = []
        for cust in vmi_customers:
            # Each VMI customer linked to 1-2 VMI plants
            n_plants = int(self.rng.integers(1, min(3, len(vmi_plant_ids) + 1)))
            linked_plants = self.rng.choice(vmi_plant_ids, size=n_plants, replace=False)
            for plant in linked_plants:
                rows.append({"Plant Code": plant, "Customers": cust})

        return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

    # ── 3. PlantPorts ────────────────────────────────────────────────────────

    def generate_plant_ports(self) -> pd.DataFrame:
        """Connect each plant to 1–3 ports (PlantPorts sheet)."""
        rows = []
        for plant in self.plant_ids:
            n_links = int(self.rng.integers(1, min(4, self.config.n_ports + 1)))
            ports = self.rng.choice(self.port_ids, size=n_links, replace=False)
            for port in ports:
                rows.append({"Plant Code": plant, "Port": port})
        return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

    # ── 4. WhCosts ───────────────────────────────────────────────────────────

    def generate_wh_costs(self) -> pd.DataFrame:
        """Generate warehouse cost-per-unit for each plant (WhCosts sheet)."""
        cfg = self.config
        n = len(self.plant_ids)
        if cfg.wh_cost_dist == "uniform":
            costs = self.rng.uniform(cfg.wh_cost_min, cfg.wh_cost_max, size=n)
        else:  # lognormal
            costs = self.rng.lognormal(cfg.wh_cost_lognormal_mean, cfg.wh_cost_lognormal_sigma, size=n)
        return pd.DataFrame({"WH": self.plant_ids, "Cost/unit": np.round(costs, 6)})

    # ── 5. WhCapacities ──────────────────────────────────────────────────────

    def generate_wh_capacities(self, plant_daily_load: Dict[str, int]) -> pd.DataFrame:
        """Daily capacity = daily load × (1+buffer), with a floor (WhCapacities sheet)."""
        cfg = self.config
        caps = []
        for plant in self.plant_ids:
            load = plant_daily_load.get(plant, 0)
            cap = max(cfg.wh_capacity_floor, int(math.ceil(load * (1 + cfg.capacity_buffer))))
            caps.append(cap)
        return pd.DataFrame({"Plant ID": self.plant_ids, "Daily Capacity": caps})

    # ── 6. FreightRates ──────────────────────────────────────────────────────

    def generate_freight_rates(
        self,
        carrier_routes: Dict[str, Dict],
    ) -> pd.DataFrame:
        """
        Build a weight-bracket freight rate table (FreightRates sheet).

        carrier_routes: {carrier_id: {"modes": [...], "orig_ports": [...], "dest_ports": [...]}}
        """
        cfg = self.config
        # Build bracket list [(min_wt, max_wt), ...]
        brackets = []
        lower = 0.0
        for upper in cfg.weight_brackets_kg:
            brackets.append((lower, round(upper - 0.01, 2)))
            lower = float(upper)
        brackets.append((lower, 99999.99))

        mode_rate_ranges = {
            "AIR": cfg.air_rate_range,
            "GROUND": cfg.ground_rate_range,
            "SEA": cfg.sea_rate_range,
        }
        mode_min_cost_ranges = {
            "AIR": cfg.air_min_cost_range,
            "GROUND": cfg.ground_min_cost_range,
            "SEA": cfg.sea_min_cost_range,
        }
        mode_tpt_params = {
            "AIR": cfg.air_tpt_days,
            "GROUND": cfg.ground_tpt_days,
            "SEA": cfg.sea_tpt_days,
        }
        svc_by_mode = {
            "AIR": ["DTD", "DTP"],
            "GROUND": ["DTP", "CRF"],
            "SEA": ["DTD", "DTP", "CRF"],
        }

        rows = []
        for carrier_id, info in carrier_routes.items():
            for mode in info["modes"]:
                rr = mode_rate_ranges[mode]
                mcr = mode_min_cost_ranges[mode]
                tpt_mean, tpt_std = mode_tpt_params[mode]
                # Per-carrier fixed minimum cost and tpt
                min_cost = round(float(self.rng.uniform(*mcr)), 4)
                tpt = max(0, int(round(self.rng.normal(tpt_mean, tpt_std))))
                svc_codes = svc_by_mode[mode]
                for orig in info["orig_ports"]:
                    for dest in info["dest_ports"]:
                        if orig == dest:
                            continue
                        for svc in svc_codes:
                            # Rates decrease with weight (volume discounts)
                            base_rate = float(self.rng.uniform(*rr))
                            for i, (mn_wt, mx_wt) in enumerate(brackets):
                                # 5 % discount per bracket tier
                                tier_rate = round(base_rate * (0.95 ** i), 4)
                                rows.append({
                                    "Carrier": carrier_id,
                                    "orig_port_cd": orig,
                                    "dest_port_cd": dest,
                                    "minm_wgh_qty": mn_wt,
                                    "max_wgh_qty": mx_wt,
                                    "svc_cd": svc,
                                    "minimum cost": min_cost,
                                    "rate": tier_rate,
                                    "mode_dsc": mode,
                                    "tpt_day_cnt": tpt,
                                    "Carrier type": carrier_id.replace("CARRIER", "CTYPE"),
                                })
        return pd.DataFrame(rows).reset_index(drop=True)

    # ── 7. OrderList ─────────────────────────────────────────────────────────

    def generate_order_list(
        self,
        vmi_customer_ids: np.ndarray,
        vmi_plant_ids: np.ndarray,
        products_per_plant_df: pd.DataFrame,
        plant_ports_df: pd.DataFrame,
        carrier_routes: Dict[str, Dict],
    ) -> pd.DataFrame:
        """
        Generate the OrderList table.

        Each row = one order with: Order ID, Order Date, Origin Port, Carrier,
        TPT, Service Level, Ship ahead day count, Ship Late Day count,
        Customer, Product ID, Plant Code, Destination Port, Unit quantity, Weight.
        """
        cfg = self.config
        rng = self.rng

        # Pre-compute lookups
        vmi_cust_set = set(vmi_customer_ids)
        vmi_plant_set = set(vmi_plant_ids)

        prod_to_plants = (
            products_per_plant_df.groupby("Product ID")["Plant Code"].apply(set).to_dict()
        )
        vmi_eligible_prods = [
            p for p in self.product_ids
            if prod_to_plants.get(p, set()) & vmi_plant_set
        ]

        plant_to_ports = (
            plant_ports_df.groupby("Plant Code")["Port"].apply(list).to_dict()
        )

        # Carrier → mode mapping for origin port selection
        carrier_list = list(carrier_routes.keys())

        n = cfg.n_orders
        date_range_days = (pd.to_datetime(cfg.end_date) - pd.to_datetime(cfg.start_date)).days
        start_ts = pd.to_datetime(cfg.start_date)

        # Vectorised draws
        cust_indices = rng.integers(0, cfg.n_customers, size=n)
        date_offsets = rng.integers(0, date_range_days + 1, size=n)
        quantities = self._sample_quantities(n)
        unit_weights = self._sample_unit_weights(n)
        tpt_vals = rng.choice(range(5), size=n, p=cfg.tpt_probs)
        ship_ahead = rng.choice(range(7), size=n, p=cfg.ship_ahead_probs)
        ship_late = rng.choice(range(7), size=n, p=cfg.ship_late_probs)
        svc_lvls = rng.choice(self._SERVICE_LEVELS, size=n, p=cfg.service_level_probs)
        carrier_choices = rng.choice(carrier_list, size=n)

        rows = []
        for i in range(n):
            cust_id = self.customer_ids[cust_indices[i]]
            is_vmi = cust_id in vmi_cust_set

            prod_id = int(rng.choice(vmi_eligible_prods if is_vmi else self.product_ids))

            eligible_plants = list(prod_to_plants.get(prod_id, set()))
            if is_vmi:
                eligible_plants = [p for p in eligible_plants if p in vmi_plant_set]
            if not eligible_plants:
                eligible_plants = list(self.plant_ids)
            plant_code = str(rng.choice(eligible_plants))

            avail_ports = plant_to_ports.get(plant_code, list(self.port_ids))
            origin_port = str(rng.choice(avail_ports))

            # Destination: any port different from origin
            dest_options = [p for p in self.port_ids if p != origin_port]
            if not dest_options:
                dest_options = list(self.port_ids)
            dest_port = str(rng.choice(dest_options))

            carrier = carrier_choices[i]
            qty = int(quantities[i])
            wt = round(float(qty * unit_weights[i]), 5)
            order_date = start_ts + pd.Timedelta(days=int(date_offsets[i]))

            rows.append({
                "Order ID": f"{1_400_000_000 + i}.0",
                "Order Date": order_date,
                "Origin Port": origin_port,
                "Carrier": carrier,
                "TPT": int(tpt_vals[i]),
                "Service Level": str(svc_lvls[i]),
                "Ship ahead day count": int(ship_ahead[i]),
                "Ship Late Day count": int(ship_late[i]),
                "Customer": cust_id,
                "Product ID": prod_id,
                "Plant Code": plant_code,
                "Destination Port": dest_port,
                "Unit quantity": qty,
                "Weight": wt,
            })

        return pd.DataFrame(rows)

    # ── Carrier route builder ─────────────────────────────────────────────────

    def _build_carrier_routes(self) -> Dict[str, Dict]:
        """
        Assign each carrier: mode(s), origin ports, and destination ports.
        Returns dict: {carrier_id: {"modes": [...], "orig_ports": [...], "dest_ports": [...]}}
        """
        cfg = self.config
        routes = {}
        modes = self._MODES
        for carrier in self.carrier_ids:
            # Primary mode
            primary = str(self.rng.choice(modes, p=cfg.carrier_mode_probs))
            carrier_modes = [primary]
            # Optionally add a second mode
            if self.rng.random() < cfg.carrier_multimodal_prob and len(modes) > 1:
                others = [m for m in modes if m != primary]
                carrier_modes.append(str(self.rng.choice(others)))

            mn_ports, mx_ports = cfg.ports_per_carrier_range
            n_orig = int(self.rng.integers(mn_ports, mx_ports + 1))
            orig_ports = list(self.rng.choice(self.port_ids, size=min(n_orig, len(self.port_ids)),
                                              replace=False))
            # Destination ports = all ports except the chosen origins (at least one)
            dest_candidates = [p for p in self.port_ids if p not in orig_ports]
            if not dest_candidates:
                dest_candidates = list(self.port_ids)
            n_dest = int(self.rng.integers(1, min(4, len(dest_candidates) + 1)))
            dest_ports = list(self.rng.choice(dest_candidates, size=n_dest, replace=False))

            routes[carrier] = {
                "modes": carrier_modes,
                "orig_ports": orig_ports,
                "dest_ports": dest_ports,
            }
        return routes

    # ── Main orchestrator ─────────────────────────────────────────────────────

    def run(self) -> Dict[str, pd.DataFrame]:
        """
        Generate all seven tables. Returns a dict keyed by the original
        Kaggle sheet names:
          OrderList, FreightRates, WhCosts, WhCapacities,
          ProductsPerPlant, VmiCustomers, PlantPorts
        """
        print("[1/7] ProductsPerPlant ...")
        products_per_plant = self.generate_products_per_plant(self.vmi_plant_ids)

        print("[2/7] VmiCustomers ...")
        vmi_customers = self.generate_vmi_customers(self.vmi_plant_ids)
        vmi_customer_ids = vmi_customers["Customers"].unique()

        print("[3/7] PlantPorts ...")
        plant_ports = self.generate_plant_ports()

        print("[4/7] WhCosts ...")
        wh_costs = self.generate_wh_costs()

        print("[5/7] FreightRates ...")
        carrier_routes = self._build_carrier_routes()
        freight_rates = self.generate_freight_rates(carrier_routes)

        print("[6/7] OrderList ...")
        order_list = self.generate_order_list(
            vmi_customer_ids=vmi_customer_ids,
            vmi_plant_ids=self.vmi_plant_ids,
            products_per_plant_df=products_per_plant,
            plant_ports_df=plant_ports,
            carrier_routes=carrier_routes,
        )

        print("[7/7] WhCapacities ...")
        n_days = max(1, (pd.to_datetime(self.config.end_date)
                        - pd.to_datetime(self.config.start_date)).days)
        plant_total_qty = order_list.groupby("Plant Code")["Unit quantity"].sum().to_dict()
        plant_daily_load = {p: int(math.ceil(v / n_days))
                            for p, v in plant_total_qty.items()}
        wh_capacities = self.generate_wh_capacities(plant_daily_load)

        return {
            "OrderList": order_list,
            "FreightRates": freight_rates,
            "WhCosts": wh_costs,
            "WhCapacities": wh_capacities,
            "ProductsPerPlant": products_per_plant,
            "VmiCustomers": vmi_customers,
            "PlantPorts": plant_ports,
        }

    # ── Optional: save to disk ────────────────────────────────────────────────

    def save(self, tables: Dict[str, pd.DataFrame]) -> Dict[str, Path]:
        """Write all tables to the configured output directory."""
        out = Path(self.config.output_dir)
        out.mkdir(parents=True, exist_ok=True)
        paths = {}
        for name, df in tables.items():
            if self.config.output_format == "csv":
                p = out / f"{name}.csv"
                df.to_csv(p, index=False)
            else:
                try:
                    import pyarrow  # noqa: F401
                    p = out / f"{name}.parquet"
                    df.to_parquet(p, index=False)
                except ImportError:
                    p = out / f"{name}.csv"
                    df.to_csv(p, index=False)
                    print(f"pyarrow not installed — saved {name} as CSV instead.")
            paths[name] = p
            print(f"  Saved {name} → {p}")
        return paths


print("SupplyChainDataGenerator class defined.")

SupplyChainDataGenerator class defined.


## 4. Validation suite

In [ ]:
def run_validation(tables: Dict[str, pd.DataFrame], config: GeneratorConfig) -> None:
    """
    Comprehensive integrity checks on all seven generated tables.
    Raises AssertionError on the first failure; prints a summary on success.
    """
    orders = tables["OrderList"]
    freight = tables["FreightRates"]
    wh_costs = tables["WhCosts"]
    wh_caps = tables["WhCapacities"]
    ppp = tables["ProductsPerPlant"]
    vmi_cust = tables["VmiCustomers"]
    pp = tables["PlantPorts"]

    errors = []

    # ── OrderList ──────────────────────────────────────────────────────────
    # 1. Row count
    if len(orders) != config.n_orders:
        errors.append(f"OrderList has {len(orders)} rows, expected {config.n_orders}")

    # 2. No nulls in critical columns
    critical_cols = ["Order ID", "Order Date", "Customer", "Product ID",
                     "Plant Code", "Unit quantity", "Weight", "Carrier"]
    for col in critical_cols:
        nulls = orders[col].isna().sum()
        if nulls > 0:
            errors.append(f"OrderList['{col}'] has {nulls} nulls")

    # 3. Quantity and weight must be positive
    if (orders["Unit quantity"] <= 0).any():
        errors.append("OrderList: Unit quantity has non-positive values")
    if (orders["Weight"] <= 0).any():
        errors.append("OrderList: Weight has non-positive values")

    # 4. Order dates within configured range
    s, e = pd.to_datetime(config.start_date), pd.to_datetime(config.end_date)
    if (orders["Order Date"] < s).any() or (orders["Order Date"] > e).any():
        errors.append("OrderList: Order Date out of configured range")

    # 5. Service levels are valid
    valid_svc = {"DTP", "DTD", "CRF"}
    invalid_svc = set(orders["Service Level"].unique()) - valid_svc
    if invalid_svc:
        errors.append(f"OrderList: Unknown Service Level values: {invalid_svc}")

    # 6. All Plant Codes in orders exist in ProductsPerPlant
    order_plants = set(orders["Plant Code"].unique())
    ppp_plants = set(ppp["Plant Code"].unique())
    unknown_plants = order_plants - ppp_plants
    if unknown_plants:
        errors.append(f"OrderList: Plant Codes not in ProductsPerPlant: {unknown_plants}")

    # 7. VMI customers only served by VMI plants
    vmi_cust_set = set(vmi_cust["Customers"].unique())
    vmi_plant_set = set(vmi_cust["Plant Code"].unique())
    vmi_orders = orders[orders["Customer"].isin(vmi_cust_set)]
    bad_vmi = vmi_orders[~vmi_orders["Plant Code"].isin(vmi_plant_set)]
    if len(bad_vmi) > 0:
        errors.append(f"OrderList: {len(bad_vmi)} VMI orders served by non-VMI plants")

    # 8. Every product in orders has a plant in ProductsPerPlant
    order_prods = set(orders["Product ID"].unique())
    ppp_prods = set(ppp["Product ID"].unique())
    missing_prods = order_prods - ppp_prods
    if missing_prods:
        errors.append(f"OrderList: {len(missing_prods)} ordered products have no plant coverage")

    # ── FreightRates ───────────────────────────────────────────────────────
    # 9. No negative rates or costs
    if (freight["rate"] < 0).any():
        errors.append("FreightRates: Negative rate values found")
    if (freight["minimum cost"] < 0).any():
        errors.append("FreightRates: Negative minimum cost values found")

    # 10. Weight brackets are non-overlapping within each carrier–route–mode–svc group
    fr_key = ["Carrier", "orig_port_cd", "dest_port_cd", "mode_dsc", "svc_cd"]
    dup_brackets = freight.duplicated(subset=fr_key + ["minm_wgh_qty"])
    if dup_brackets.any():
        errors.append(f"FreightRates: {dup_brackets.sum()} duplicate weight-bracket entries")

    # 11. mode_dsc only contains valid modes
    valid_modes = {"AIR", "GROUND", "SEA"}
    invalid_modes = set(freight["mode_dsc"].unique()) - valid_modes
    if invalid_modes:
        errors.append(f"FreightRates: Unknown mode_dsc values: {invalid_modes}")

    # 12. tpt_day_cnt non-negative
    if (freight["tpt_day_cnt"] < 0).any():
        errors.append("FreightRates: Negative tpt_day_cnt")

    # ── WhCosts / WhCapacities ─────────────────────────────────────────────
    # 13. All n_plants represented
    if len(wh_costs) != config.n_plants:
        errors.append(f"WhCosts: {len(wh_costs)} rows, expected {config.n_plants}")
    if len(wh_caps) != config.n_plants:
        errors.append(f"WhCapacities: {len(wh_caps)} rows, expected {config.n_plants}")

    # 14. Positive costs and capacities
    if (wh_costs["Cost/unit"] <= 0).any():
        errors.append("WhCosts: Non-positive Cost/unit values")
    if (wh_caps["Daily Capacity"] < config.wh_capacity_floor).any():
        errors.append(f"WhCapacities: Capacity below floor ({config.wh_capacity_floor})")

    # ── ProductsPerPlant ───────────────────────────────────────────────────
    # 15. All products covered
    all_prods_set = set(range(config.product_id_start,
                              config.product_id_start + config.n_products))
    if not all_prods_set.issubset(ppp_prods):
        errors.append(f"ProductsPerPlant: {len(all_prods_set - ppp_prods)} products have no plant")

    # 16. No duplicates
    dup_ppp = ppp.duplicated(subset=["Plant Code", "Product ID"]).sum()
    if dup_ppp > 0:
        errors.append(f"ProductsPerPlant: {dup_ppp} duplicate rows")

    # ── VmiCustomers ───────────────────────────────────────────────────────
    # 17. VMI plants are a subset of all plants
    all_plants = set(f"{config.plant_prefix}_{i:02d}" for i in range(1, config.n_plants + 1))
    vmi_plants_in_table = set(vmi_cust["Plant Code"].unique())
    bad_vmi_plants = vmi_plants_in_table - all_plants
    if bad_vmi_plants:
        errors.append(f"VmiCustomers: Unknown Plant Codes: {bad_vmi_plants}")

    # 18. VMI customers are a subset of all customers
    all_custs = set(f"{config.customer_prefix}_{i:03d}" for i in range(1, config.n_customers + 1))
    bad_custs = set(vmi_cust["Customers"].unique()) - all_custs
    if bad_custs:
        errors.append(f"VmiCustomers: Unknown Customer IDs: {bad_custs}")

    # ── PlantPorts ─────────────────────────────────────────────────────────
    # 19. Every plant has at least one port
    plants_with_ports = set(pp["Plant Code"].unique())
    plants_without_port = all_plants - plants_with_ports
    if plants_without_port:
        errors.append(f"PlantPorts: Plants with no port link: {plants_without_port}")

    # ── Report ─────────────────────────────────────────────────────────────
    if errors:
        msg = "\n  ❌ ".join(errors)
        raise AssertionError("Validation FAILED:\n  ❌ " + msg)

    print("✅  All 19 validation checks passed.")
    print(f"    OrderList       : {len(orders):>8,} rows")
    print(f"    FreightRates    : {len(freight):>8,} rows")
    print(f"    WhCosts         : {len(wh_costs):>8,} rows")
    print(f"    WhCapacities    : {len(wh_caps):>8,} rows")
    print(f"    ProductsPerPlant: {len(ppp):>8,} rows")
    print(f"    VmiCustomers    : {len(vmi_cust):>8,} rows")
    print(f"    PlantPorts      : {len(pp):>8,} rows")


print("Validation function defined.")

Validation function defined.


## 5. Run the generator

Edit `config` below to customise your dataset. Sensible defaults are pre-filled.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  ✏️  EDIT THIS CELL to configure your dataset
# ─────────────────────────────────────────────────────────────────────────────

config = GeneratorConfig(
    # ── Scale
    n_orders=10_000,
    n_customers=500,
    n_products=200,
    n_plants=20,
    n_ports=8,
    n_carriers=12,

    # ── Time range
    start_date="2026-01-01",
    end_date="2026-12-31",

    # ── VMI settings
    # vmi_plant_ids=("PLANT_02", "PLANT_06", "PLANT_10", "PLANT_11"),
    vmi_plant_ids=None,       # None → auto-select ~20 % of plants
    vmi_customer_ratio=0.15,

    # ── Product–plant coverage
    min_products_per_plant=0.30,
    max_products_per_plant=0.70,

    # ── Capacity
    capacity_buffer=0.20,

    # ── Order quantity distribution
    # Options: "uniform" | "lognormal" | "zipf"
    order_qty_dist="uniform",
    min_order_qty=5,
    max_order_qty=50,
    # lognormal params (only used when order_qty_dist="lognormal")
    order_qty_lognormal_mean=3.5,
    order_qty_lognormal_sigma=0.8,

    # ── Weight per unit
    # Options: "uniform" | "gamma"
    weight_per_unit_dist="uniform",
    weight_per_unit_min=0.1,
    weight_per_unit_max=5.0,

    # ── Service level mix  [DTP, DTD, CRF]
    service_level_probs=(0.67, 0.23, 0.10),

    # ── TPT distribution   [0d, 1d, 2d, 3d, 4d]
    tpt_probs=(0.05, 0.23, 0.70, 0.01, 0.01),

    # ── Ship-ahead / late  [0..6 days]
    ship_ahead_probs=(0.25, 0.05, 0.05, 0.40, 0.10, 0.10, 0.05),
    ship_late_probs=(0.95, 0.01, 0.01, 0.01, 0.01, 0.005, 0.005),

    # ── Carrier mode mix   [AIR, GROUND, SEA]
    carrier_mode_probs=(0.50, 0.30, 0.20),
    carrier_multimodal_prob=0.25,

    # ── Carrier port coverage
    ports_per_carrier_range=(1, 4),

    # ── Freight rate weight brackets (kg upper bounds)
    weight_brackets_kg=(5, 10, 25, 50, 100, 300, 500, 1000, 5000),

    # ── Warehouse cost distribution
    # Options: "uniform" | "lognormal"
    wh_cost_dist="lognormal",
    wh_cost_lognormal_mean=0.0,
    wh_cost_lognormal_sigma=0.6,

    # ── Output
    output_dir="supply_chain_output",
    output_format="csv",
    seed=42,
)

print("Config ready. Generating ...")
gen = SupplyChainDataGenerator(config)
tables = gen.run()

print("\nRunning validation ...")
run_validation(tables, config)

Config ready. Generating ...
[1/7] ProductsPerPlant ...
[2/7] VmiCustomers ...
[3/7] PlantPorts ...
[4/7] WhCosts ...
[5/7] FreightRates ...
[6/7] OrderList ...
[7/7] WhCapacities ...

Running validation ...
✅  All 19 validation checks passed.
    OrderList       :   10,000 rows
    FreightRates    :    1,610 rows
    WhCosts         :       20 rows
    WhCapacities    :       20 rows
    ProductsPerPlant:    1,932 rows
    VmiCustomers    :      113 rows
    PlantPorts      :       42 rows


## 6. Quick EDA previews

All seven tables are in `tables["<SheetName>"]` as DataFrames.

In [ ]:
# ── Table summaries ──────────────────────────────────────────────────────────
for name, df in tables.items():
    print(f"{'='*60}")
    print(f"  {name}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
    print(f"{'='*60}")
    display(df.head(5))
    print()

  OrderList  (10,000 rows × 14 cols)


,Order ID,Order Date,Origin Port,Carrier,TPT,Service Level,Ship ahead day count,Ship Late Day count,Customer,Product ID,Plant Code,Destination Port,Unit quantity,Weight
0,1400000000.0,2026-09-25,PORT08,CARRIER_08,4,DTD,0,0,CUST_346,1600159,PLANT_08,PORT06,41,145.19457
1,1400000001.0,2026-02-26,PORT07,CARRIER_10,1,DTP,3,0,CUST_270,1600167,PLANT_07,PORT03,15,60.38501
2,1400000002.0,2026-09-19,PORT05,CARRIER_10,2,CRF,4,0,CUST_169,1600063,PLANT_12,PORT04,10,21.10427
3,1400000003.0,2026-01-18,PORT07,CARRIER_08,2,DTP,3,0,CUST_241,1600023,PLANT_13,PORT02,27,48.57371
4,1400000004.0,2026-09-27,PORT01,CARRIER_04,1,DTD,3,0,CUST_420,1600142,PLANT_01,PORT04,42,63.58392



  FreightRates  (1,610 rows × 11 cols)


,Carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum cost,rate,mode_dsc,tpt_day_cnt,Carrier type
0,CARRIER_01,PORT02,PORT08,0.0,4.99,DTD,78.0515,2.2049,AIR,2,CTYPE_01
1,CARRIER_01,PORT02,PORT08,5.0,9.99,DTD,78.0515,2.0947,AIR,2,CTYPE_01
2,CARRIER_01,PORT02,PORT08,10.0,24.99,DTD,78.0515,1.9899,AIR,2,CTYPE_01
3,CARRIER_01,PORT02,PORT08,25.0,49.99,DTD,78.0515,1.8904,AIR,2,CTYPE_01
4,CARRIER_01,PORT02,PORT08,50.0,99.99,DTD,78.0515,1.7959,AIR,2,CTYPE_01



  WhCosts  (20 rows × 2 cols)


...

,Plant Code,Product ID
0,PLANT_01,1600176
1,PLANT_01,1600141
2,PLANT_01,1600146
3,PLANT_01,1600036
4,PLANT_01,1600054



  VmiCustomers  (113 rows × 2 cols)


,Plant Code,Customers
0,PLANT_02,CUST_241
1,PLANT_02,CUST_496
2,PLANT_09,CUST_496
3,PLANT_02,CUST_426
4,PLANT_14,CUST_426



  PlantPorts  (42 rows × 2 cols)


,Plant Code,Port
0,PLANT_01,PORT02
1,PLANT_01,PORT01
2,PLANT_02,PORT04
3,PLANT_02,PORT02
4,PLANT_02,PORT06


In [ ]:
# ── OrderList — descriptive stats ────────────────────────────────────────────
orders = tables["OrderList"]
print("OrderList — numeric summary")
display(orders[["Unit quantity", "Weight", "TPT",
               "Ship ahead day count", "Ship Late Day count"]].describe().round(3))

print("\nService Level distribution")
display(orders["Service Level"].value_counts(normalize=True).rename("fraction").round(3))

print("\nTPT distribution")
display(orders["TPT"].value_counts().sort_index())

OrderList — numeric summary


,Unit quantity,Weight,TPT,Ship ahead day count,Ship Late Day count
count,10000.000,10000.000,10000.000,10000.000,10000.000
mean,27.675,70.824,1.695,2.567,0.150
std,13.345,54.998,0.620,1.798,0.741
min,5.000,0.550,0.000,0.000,0.000
25%,16.000,26.004,1.000,1.000,0.000
50%,28.000,56.788,2.000,3.000,0.000
75%,39.000,104.694,2.000,4.000,0.000
max,50.000,248.033,4.000,6.000,6.000



Service Level distribution


,fraction
Service Level,
DTP,0.662
DTD,0.235
CRF,0.103



TPT distribution


,count
TPT,
0,501
1,2324
2,6988
3,101
4,86


In [ ]:
# ── FreightRates — mode and bracket coverage ─────────────────────────────────
fr = tables["FreightRates"]
print("Carrier count by mode")
display(fr.groupby("mode_dsc")["Carrier"].nunique().rename("n_carriers"))

print("\nRate statistics by mode")
display(fr.groupby("mode_dsc")["rate"].agg(["min", "median", "max"]).round(4))

print("\nTransit-day distribution by mode")
display(fr.groupby("mode_dsc")["tpt_day_cnt"].value_counts().rename("count"))

Carrier count by mode


,n_carriers
mode_dsc,
AIR,7
GROUND,5
SEA,4



Rate statistics by mode


,min,median,max
mode_dsc,,,
AIR,0.3396,2.2294,4.9895
GROUND,0.0367,0.3114,0.7934
SEA,0.0184,0.1314,0.2986



Transit-day distribution by mode


mode_dsc  tpt_day_cnt
AIR       2              200
          1              160
          3              160
          0              120
GROUND    4              220
          3              120
          6              120
SEA       18             450
          14              60
Name: count, dtype: int64

In [ ]:
# ── Warehouse costs and capacities ───────────────────────────────────────────
print("WhCosts — Cost/unit summary")
display(tables["WhCosts"]["Cost/unit"].describe().round(4))

print("\nWhCapacities — Daily Capacity summary")
display(tables["WhCapacities"]["Daily Capacity"].describe())

WhCosts — Cost/unit summary


,Cost/unit
count,20.0000
mean,1.1654
std,0.4329
min,0.5335
25%,0.8534
50%,1.1309
75%,1.3590
max,2.3176



WhCapacities — Daily Capacity summary


,Daily Capacity
count,20.000000
mean,46.650000
std,17.254366
min,22.000000
25%,36.000000
50%,46.000000
75%,56.000000
max,99.000000


In [ ]:
# ── VMI coverage ─────────────────────────────────────────────────────────────
vmi = tables["VmiCustomers"]
print(f"VMI plants : {vmi['Plant Code'].nunique()}")
print(f"VMI customers: {vmi['Customers'].nunique()}")
print(f"Fraction VMI orders: "
      f"{orders['Customer'].isin(vmi['Customers'].unique()).mean():.1%}")
display(vmi.head(10))

VMI plants : 4
VMI customers: 75
Fraction VMI orders: 15.3%


,Plant Code,Customers
0,PLANT_02,CUST_241
1,PLANT_02,CUST_496
2,PLANT_09,CUST_496
3,PLANT_02,CUST_426
4,PLANT_14,CUST_426
5,PLANT_02,CUST_212
6,PLANT_13,CUST_076
7,PLANT_02,CUST_076
8,PLANT_09,CUST_128
9,PLANT_14,CUST_277


## 7. Save to disk (optional)

In [ ]:
paths = gen.save(tables)
print("Files saved:")
for name, p in paths.items():
  print(f"  {name}: {p}")

  Saved OrderList → supply_chain_output/OrderList.csv
  Saved FreightRates → supply_chain_output/FreightRates.csv
  Saved WhCosts → supply_chain_output/WhCosts.csv
  Saved WhCapacities → supply_chain_output/WhCapacities.csv
  Saved ProductsPerPlant → supply_chain_output/ProductsPerPlant.csv
  Saved VmiCustomers → supply_chain_output/VmiCustomers.csv
  Saved PlantPorts → supply_chain_output/PlantPorts.csv
Files saved:
  OrderList: supply_chain_output/OrderList.csv
  FreightRates: supply_chain_output/FreightRates.csv
  WhCosts: supply_chain_output/WhCosts.csv
  WhCapacities: supply_chain_output/WhCapacities.csv
  ProductsPerPlant: supply_chain_output/ProductsPerPlant.csv
  VmiCustomers: supply_chain_output/VmiCustomers.csv
  PlantPorts: supply_chain_output/PlantPorts.csv


## 8. (Optional) Alternate configuration examples

Copy any block below and rerun the generator.

In [ ]:
'''# ── Example A: Right-skewed (lognormal) order quantities ─────────────────────
config_lognormal = GeneratorConfig(
    n_orders=5_000,
    n_customers=200,
    n_products=100,
    n_plants=10,
    n_ports=6,
    n_carriers=8,
    order_qty_dist="lognormal",
    order_qty_lognormal_mean=3.5,
    order_qty_lognormal_sigma=1.0,
    min_order_qty=1,
    max_order_qty=500,
    weight_per_unit_dist="gamma",
    weight_per_unit_gamma_shape=2.0,
    weight_per_unit_gamma_scale=1.5,
    wh_cost_dist="lognormal",
    service_level_probs=(0.60, 0.30, 0.10),
    carrier_mode_probs=(0.40, 0.40, 0.20),
    seed=99,
)

gen_ln = SupplyChainDataGenerator(config_lognormal)
tables_ln = gen_ln.run()
run_validation(tables_ln, config_lognormal)

print("\nQuantity distribution (lognormal config):")
print(tables_ln["OrderList"]["Unit quantity"].describe().round(1))'''

'# ── Example A: Right-skewed (lognormal) order quantities ─────────────────────\nconfig_lognormal = GeneratorConfig(\n    n_orders=5_000,\n    n_customers=200,\n    n_products=100,\n    n_plants=10,\n    n_ports=6,\n    n_carriers=8,\n    order_qty_dist="lognormal",\n    order_qty_lognormal_mean=3.5,\n    order_qty_lognormal_sigma=1.0,\n    min_order_qty=1,\n    max_order_qty=500,\n    weight_per_unit_dist="gamma",\n    weight_per_unit_gamma_shape=2.0,\n    weight_per_unit_gamma_scale=1.5,\n    wh_cost_dist="lognormal",\n    service_level_probs=(0.60, 0.30, 0.10),\n    carrier_mode_probs=(0.40, 0.40, 0.20),\n    seed=99,\n)\n\ngen_ln = SupplyChainDataGenerator(config_lognormal)\ntables_ln = gen_ln.run()\nrun_validation(tables_ln, config_lognormal)\n\nprint("\nQuantity distribution (lognormal config):")\nprint(tables_ln["OrderList"]["Unit quantity"].describe().round(1))'